In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Analyze store data with the BigQuery tools

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### The BigQuery toolset

ADK ships a [BigQuery toolset](https://adk.dev/integrations/bigquery/) that gives an agent a set of ready-made tools: list tables, read a table's schema, run SQL, ask questions in natural language with Conversational Analytics (`ask_data_insights`), `forecast` a time series and `detect_anomalies`. You choose which tools the agent gets and configure how they may be used.

### Governed access

An analyst agent must not change data or read data it should not see. This quickstart controls that in three places:

- `BigQueryToolConfig`: `WriteMode.BLOCKED` refuses anything but reads, with caps on rows and bytes billed, and job labels that make every query traceable in `INFORMATION_SCHEMA.JOBS`.
- A `before_tool_callback` that refuses any call that mentions the people tables (associates and coaching signals).
- A second callback that allows only `SELECT` statements against your own dataset.

<img width="60%" src="../../docs/diagrams/q05.png" alt="An analyst agent that queries the store dataset through the BigQuery toolset, with write and people-data guards" />

### Objectives

In this tutorial, you will learn how to give an agent governed, read-only analytical access to a BigQuery dataset.

You will complete the following tasks:

- Look at the store dataset with the BigQuery client
- Configure the BigQuery toolset for read-only use
- Add callbacks that block people data and anything outside your dataset
- Ask the agent analytical questions, including a forecast

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Gemini on Vertex AI pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing), [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded during setup, in your own namespace. Set your project ID and the namespace you chose.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# This notebook sits two folders below the repository root, where the shared store tools live
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, SDKs announce renamed classes with a FutureWarning, and the Gen AI SDK logs a note
# whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

import json
import re

import google.auth
from google.adk.agents import LlmAgent
from google.adk.integrations.bigquery import BigQueryCredentialsConfig, BigQueryToolset
from google.adk.integrations.bigquery.config import BigQueryToolConfig, WriteMode
from google.adk.models import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import BaseTool, ToolContext
from google.cloud import bigquery
from google.genai import types

from agents.cymbal_store_ops.tools.sql_guard import SqlGuardError, assert_select_only

### Choose the model

The agent in this tutorial uses Gemini 3.8 Flash. The retry options make the SDK retry a request that fails with a temporary error, such as a 429 or a 500, instead of failing the turn.

In [4]:
model = Gemini(
    model="gemini-3.8-flash",
    retry_options=types.HttpRetryOptions(attempts=4, initial_delay=2.0),
)

## Look at the dataset

Your store data is in the dataset `cymbal_beauty_<namespace>_dev`. List its tables and their row counts with the BigQuery client:

In [5]:
DATASET = f"cymbal_beauty_{WORKSHOP_NAMESPACE}_dev"
bq = bigquery.Client(project=PROJECT_ID)

for table in bq.list_tables(DATASET):
    print(f"{table.table_id:24} {bq.get_table(table).num_rows:>8,} rows")

associates                    200 rows
bopis_orders                2,000 rows


coaching_signals              624 rows
guest_feedback                800 rows


operations_context             16 rows
products                      600 rows


replenishment               1,200 rows
reviews                     6,000 rows


shrink_events               1,500 rows


store_inventory            24,000 rows


store_tasks                   400 rows


store_traffic               6,720 rows
stores                         40 rows


Thirteen tables. `associates` and `coaching_signals` hold people data; the callbacks below keep the agent away from them.

## Configure the toolset

`tool_filter` picks the six tools the agent gets. `BigQueryToolConfig` blocks writes, caps each result at 50 rows and 100 MB billed, bills queries to your project, and labels every job with the agent's name.

In [6]:
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])

toolset = BigQueryToolset(
    tool_filter=[
        "list_table_ids",
        "get_table_info",
        "execute_sql",
        "ask_data_insights",
        "forecast",
        "detect_anomalies",
    ],
    credentials_config=BigQueryCredentialsConfig(credentials=credentials),
    bigquery_tool_config=BigQueryToolConfig(
        write_mode=WriteMode.BLOCKED,
        max_query_result_rows=50,
        maximum_bytes_billed=100 * 1024 * 1024,
        compute_project_id=PROJECT_ID,
        location="US",
        application_name="data_analyst_agent",
        job_labels={"adk_agent": "data_analyst_agent", "ns": WORKSHOP_NAMESPACE},
    ),
)

## Add the guard callbacks

A `before_tool_callback` runs before every tool call. If it returns a dictionary, ADK skips the tool and gives that dictionary to the model as the result.

`block_people_data` refuses any call whose arguments mention the associates or coaching signals tables. People data goes through the store agent's role-checked tools, never raw SQL.

In [7]:
PEOPLE_TABLES = re.compile(r"\b(associates|coaching_signals)\b", re.IGNORECASE)


def block_people_data(tool: BaseTool, args: dict, tool_context: ToolContext) -> dict | None:
    """Refuse any tool call that names a people table."""
    if PEOPLE_TABLES.search(json.dumps(args, default=str)):
        return {"status": "ERROR", "error_details": "blocked by policy: people data is not available to the analyst"}
    return None

`only_this_dataset` allows metadata calls for your dataset only, and SQL only when it is a single `SELECT` that reads tables in your dataset. `assert_select_only` parses the statement to check that.

In [8]:
def only_this_dataset(tool: BaseTool, args: dict, tool_context: ToolContext) -> dict | None:
    """Allow SELECT statements and metadata calls for this dataset only."""
    if tool.name in ("list_table_ids", "get_table_info"):
        if args.get("dataset_id", DATASET) != DATASET:
            return {"status": "ERROR", "error_details": f"blocked by policy: only {DATASET} is available"}
    if tool.name == "execute_sql":
        try:
            assert_select_only(args.get("query", ""), (f"{PROJECT_ID}.{DATASET}",))
        except SqlGuardError as error:
            return {"status": "ERROR", "error_details": f"blocked by policy: {error}"}
    return None


only_this_dataset(
    BaseTool(name="execute_sql", description=""),
    {"query": f"DELETE FROM `{PROJECT_ID}.{DATASET}.shrink_events` WHERE TRUE"},
    None,
)

{'status': 'ERROR',
 'error_details': 'blocked by policy: only SELECT/WITH queries are allowed (statement starts with DELETE)'}

The callback refuses the `DELETE` before anything reaches BigQuery. This cell calls the callback directly, without a model, to show the check.

## Define the agent

The instruction describes the dataset and says how to answer: numbers from the tools, then the SQL that produced them.

In [9]:
instruction = f"""You are the store operations data analyst for Cymbal Beauty district and store managers.
The data is in the BigQuery dataset `{PROJECT_ID}.{DATASET}`: stores, products, store_inventory,
bopis_orders, store_traffic, shrink_events, store_tasks, replenishment, guest_feedback and reviews.
store_traffic has one row per store and open hour (store_id, ts_hour in UTC, visitors, transactions,
sales_usd); stores keep America/Chicago time, so a daily series sums visitors by DATE(ts_hour, 'America/Chicago').
People data (associates, coaching signals) is not available here.
- Read a table's columns with get_table_info before writing SQL against it.
- Answer with numbers from execute_sql. Use forecast for questions about the coming days on a time
  series, and detect_anomalies for questions about anything unusual.
- Answer in one or two sentences, then one line starting "SQL:" with the query you ran;
  when you ran no query, give no SQL line.
- Read-only: never attempt INSERT, UPDATE, DELETE or DDL."""

agent = LlmAgent(
    name="data_analyst_agent",
    model=model,
    description="Answers store operations questions with read-only BigQuery tools.",
    instruction=instruction,
    tools=[toolset],
    before_tool_callback=[block_people_data, only_this_dataset],
)

## Run the agent

Create a runner and a session:

In [10]:
runner = InMemoryRunner(agent=agent, app_name="data_analyst_agent")
session = await runner.session_service.create_session(app_name="data_analyst_agent", user_id="manager")

Define a helper that sends one message and prints what happened: each tool call with its arguments, and the agent's reply.

In [11]:
async def ask(question: str) -> None:
    """Send one message to the agent and print its tool calls and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            print(f"[{event.author}] calls {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n{text}\n")

Ask an aggregate question. The agent reads the schema, writes SQL and answers with the query it ran:

In [12]:
await ask("Which product category has the highest total shrink value across all stores?")

[data_analyst_agent] calls get_table_info({'project_id': 'mattrobn-sandbox', 'dataset_id': 'cymbal_beauty_opsreview_dev', 'table_id': 'shrink_events'})
[data_analyst_agent] calls get_table_info({'table_id': 'products', 'project_id': 'mattrobn-sandbox', 'dataset_id': 'cymbal_beauty_opsreview_dev'})


[data_analyst_agent] calls execute_sql({'query': 'SELECT p.category, ROUND(SUM(s.value_usd), 2) AS total_shrink_usd\nFROM `mattrobn-sandbox.cymbal_beauty_opsreview_dev.shrink_events` s\nJOIN `mattrobn-sandbox.cymbal_beauty_opsreview_dev.products` p\n  ON s.product_id = p.product_id\nGROUP BY p.category\nORDER BY total_shrink_usd DESC', 'project_id': 'mattrobn-sandbox'})



The skincare category has the highest total shrink value across all stores at $21,973.45.

SQL: SELECT p.category, ROUND(SUM(s.value_usd), 2) AS total_shrink_usd FROM `mattrobn-sandbox.cymbal_beauty_opsreview_dev.shrink_events` s JOIN `mattrobn-sandbox.cymbal_beauty_opsreview_dev.products` p ON s.product_id = p.product_id GROUP BY p.category ORDER BY total_shrink_usd DESC



The agent read the two schemas, joined `shrink_events` to `products`, and found skincare highest at $21,973.45. The SQL can differ from run to run; the total should not.

Ask for a forecast. The agent sums the hourly traffic into days and calls the `forecast` tool, which runs BigQuery's `AI.FORECAST` on that daily series. The instruction's line about `store_traffic` tells it how to build the daily series. This cell takes about 30 seconds:

In [13]:
await ask("Forecast daily visitors at store S-014 for the next 7 days.")

[data_analyst_agent] calls get_table_info({'project_id': 'mattrobn-sandbox', 'table_id': 'store_traffic', 'dataset_id': 'cymbal_beauty_opsreview_dev'})


[data_analyst_agent] calls execute_sql({'query': "SELECT \n  DATE(ts_hour, 'America/Chicago') as date,\n  SUM(visitors) as daily_visitors\nFROM `mattrobn-sandbox.cymbal_beauty_opsreview_dev.store_traffic`\nWHERE store_id = 'S-014'\nGROUP BY date\nORDER BY date", 'project_id': 'mattrobn-sandbox'})


[data_analyst_agent] calls forecast({'timestamp_col': 'traffic_date', 'data_col': 'visitors', 'horizon': 7, 'project_id': 'mattrobn-sandbox', 'history_data': "SELECT TIMESTAMP(DATE(ts_hour, 'America/Chicago')) AS traffic_date, SUM(visitors) AS visitors FROM `mattrobn-sandbox.cymbal_beauty_opsreview_dev.store_traffic` WHERE store_id = 'S-014' GROUP BY traffic_date"})



The forecasted daily visitors for store S-014 for the next 7 days (October 4–10, 2026) are approximately 614, 463, 423, 450, 457, 467, and 571 visitors, respectively.



In the run shown, the agent read the table's columns, ran the daily series with `execute_sql`, then passed the same query to `forecast`. The traffic history ends on 3 October 2026, so the forecast covers 4 to 10 October. The steps and the wording can vary from run to run.

Ask for people data. The instruction says people data is not available, and if the model tries to read it anyway, `block_people_data` refuses the call before it reaches BigQuery:

In [14]:
await ask("Which associates have the most coaching signals?")


People data, including associates and coaching signals, is not available in this dataset.



In this run the model refused from the instruction alone and made no tool call, so `block_people_data` had nothing to stop. The callback is there for the run where the model does try.

Try these example phrases:

```
How many pick-up orders did each store have last week?
Is anything unusual in daily shrink events at S-014?
Delete the shrink events for S-014
```

## Run the agent in the ADK developer UI

`agent.py` in this folder defines the same agent and reads the project and dataset from the environment. From the repository root:

```bash
uv run python scripts/quickstart_apps.py 05-data-analyst-agent
uv run adk web build/quickstart_apps --port 8001
```

## Cleaning up

This notebook creates no cloud resources. Every query it ran carries the `adk_agent: data_analyst_agent` label, so you can find them in `INFORMATION_SCHEMA.JOBS`:

In [15]:
query = """
SELECT creation_time, total_bytes_billed, LEFT(TRIM(query), 80) AS query
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
  AND EXISTS (SELECT 1 FROM UNNEST(labels) WHERE key = "adk_agent" AND value = "data_analyst_agent")
ORDER BY creation_time DESC
LIMIT 5
"""
for row in bq.query(query).result():
    print(row.creation_time, row.total_bytes_billed, row.query)

2026-09-24 00:37:53.457000+00:00 10485760 SELECT * FROM AI.FORECAST(
    (SELECT TIMESTAMP(DATE(ts_hour, 'America/Chicago'
2026-09-24 00:37:48.066000+00:00 10485760 SELECT 
  DATE(ts_hour, 'America/Chicago') as date,
  SUM(visitors) as daily_vis
2026-09-24 00:37:38.403000+00:00 20971520 SELECT p.category, ROUND(SUM(s.value_usd), 2) AS total_shrink_usd
FROM `mattrobn


These are the agent's queries from this notebook, newest first. Each is billed at least 10 MB per table it reads, BigQuery's minimum, so the join shows 20 MB.

## What's next

- [BigQuery tools in ADK](https://adk.dev/integrations/bigquery/)
- [Callbacks in ADK](https://google.github.io/adk-docs/callbacks/)
- [Quickstart 06: remember a manager's preferences with Memory Bank](../06-memory-agent/walkthrough.ipynb)